# Extracción de la API de Resultados Electorales (DINE)

Recolecta resultados de la **categoría Presidente** en **Santa Fe** desde la API
de la Dirección Nacional Electoral, al nivel más fino que cada elección permita.

Forma parte de *Análisis electoral Región Centro 2003-2023*. La salida respeta
`docs/CRITERIO_BASE_DE_DATOS.md` del repositorio.

## Antes de correrlo, cuatro cosas que conviene saber

1. **Solo hay datos desde 2011.** Verificado con `/api/menu/periodos`, que
   devuelve `[2025, 2023, 2021, 2019, 2017, 2015, 2013, 2011]`.
2. **Es recuento PROVISORIO, no escrutinio definitivo.** En el total provincial
   de la PASO 2023 la diferencia con el definitivo es de unos 37.000 votos
   (−2,1 %) en las fuerzas principales. No mezclar ambos en un mismo cálculo.
3. **Los ids de ámbito no están documentados y cambian entre años.** No hay
   endpoint que los liste (`/api/menu/distritos` devuelve error), así que hay
   que descubrirlos por barrido. Es la parte lenta.
4. **El agregado a nivel distrito está roto en algunas elecciones.** En 2011 y
   en las PASO 2023 devuelve un subconjunto en vez del total. Por eso el
   notebook siempre valida la suma de las partes contra el total declarado y
   avisa cuando no cierran.

No hace falta token: responde sin autenticación.

## 1. Configuración

In [ ]:
BASE = 'https://resultados.mininterior.gob.ar/api/resultados/getResultados'

DISTRITO_ID = '21'        # Santa Fe
CATEGORIA_ID = '1'        # Presidente y Vice
TIPO_RECUENTO = '1'       # Provisional (es el unico disponible)

INSTANCIAS = {'1': 'PASO', '2': 'GENERAL', '3': 'BALOTAJE'}
ANIOS = ['2011', '2015', '2019', '2023']

WORKERS = 16              # bajar si la API empieza a cortar conexiones
TIMEOUT = 30
REINTENTOS = 3

# Rango del barrido de codigos de circuito. Los de Santa Fe 2023 van de
# 00010 a 09999; para otros anios el rango es desconocido, asi que conviene
# empezar amplio y despues acotar con lo que se encuentre.
RANGO_CIRCUITOS = range(1, 10000)

## 2. Cliente

Una sola función hace todas las consultas. Reintenta con espera creciente,
porque la API corta conexiones cuando se la presiona.

In [ ]:
import json, time, urllib.parse, urllib.request
from concurrent.futures import ThreadPoolExecutor


def consultar(anio, tipo_eleccion, seccion=None, circuito=None, mesa=None):
    """Devuelve el JSON de un ambito, o None si no hay datos."""
    params = {
        'anioEleccion': anio, 'tipoRecuento': TIPO_RECUENTO,
        'tipoEleccion': tipo_eleccion, 'categoriaId': CATEGORIA_ID,
        'distritoId': DISTRITO_ID,
    }
    if seccion is not None:
        params['seccionId'] = str(seccion)
    if circuito is not None:
        params['circuitoId'] = str(circuito)
    if mesa is not None:
        params['mesaId'] = str(mesa)

    url = f'{BASE}?{urllib.parse.urlencode(params)}'
    for intento in range(REINTENTOS):
        try:
            with urllib.request.urlopen(url, timeout=TIMEOUT) as r:
                return json.load(r)
        except Exception:
            if intento == REINTENTOS - 1:
                return None
            time.sleep(2 ** intento)


def tiene_datos(r):
    return bool(r and r.get('estadoRecuento', {}).get('mesasTotalizadas'))


def en_paralelo(fn, items, etiqueta=''):
    """Aplica fn sobre items en paralelo, mostrando avance."""
    salida, total = [], len(items)
    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        for i, r in enumerate(ex.map(fn, items), 1):
            salida.append(r)
            if i % 250 == 0 or i == total:
                print(f'  {etiqueta} {i}/{total}', end='\r')
    print()
    return salida

## 3. Qué elecciones tienen datos

Se prueba cada combinación año × instancia y se queda con las que responden.

In [ ]:
elecciones = []
for anio in ANIOS:
    for te, nombre in INSTANCIAS.items():
        r = consultar(anio, te)
        if tiene_datos(r):
            e = r['estadoRecuento']
            elecciones.append((anio, te, nombre))
            print(f"{anio} {nombre:9} {e['mesasTotalizadas']:>6,} mesas  "
                  f"{e['cantidadElectores']:>10,} electores (agregado distrital)")
print(f'\n{len(elecciones)} elecciones con datos')

> El agregado distrital de arriba **no siempre es el total real**: en 2011 y
> en las PASO 2023 devuelve un subconjunto. Sirve para saber qué existe, no
> como cifra de control. El control verdadero es la suma de las partes.

## 4. Descubrir los ámbitos

No hay endpoint que liste secciones ni circuitos, así que se barre el espacio
de ids y se conserva lo que devuelve datos. **Es la parte lenta**: con
`RANGO_CIRCUITOS` completo son unas 10.000 consultas por elección.

Conviene correrlo primero sobre una sola elección para acotar el rango, y
recién después sobre todas.

In [ ]:
def descubrir_secciones(anio, te, maximo=120):
    out = en_paralelo(lambda s: (s, consultar(anio, te, seccion=s)),
                      list(range(1, maximo)), f'secciones {anio}')
    return [s for s, r in out if tiene_datos(r)]


def descubrir_circuitos(anio, te, rango=RANGO_CIRCUITOS):
    codigos = [f'{i:05d}' for i in rango]
    out = en_paralelo(lambda c: (c, consultar(anio, te, circuito=c)),
                      codigos, f'circuitos {anio}')
    return [c for c, r in out if tiene_datos(r)]


# Prueba sobre una eleccion conocida: Santa Fe 2023 tiene 523 circuitos.
circuitos_2023 = descubrir_circuitos('2023', '2', range(1, 10000))
print(f'2023 GENERAL: {len(circuitos_2023)} circuitos')
print('   muestra:', circuitos_2023[:8])

### Atajo: sembrar códigos ya conocidos

Para 2023 los 523 códigos de circuito de Santa Fe ya están relevados en el
repositorio (`datos/procesados/nomenclador_circuitos_2023.csv`), obtenidos del
archivo por mesa. Sembrarlos evita barrer 10.000 ids al pedo.

El barrido queda solo para 2011-2019, donde los códigos son otros por la
renumeración y todavía no los tenemos.

In [ ]:
import csv, urllib.request

RAW = ('https://raw.githubusercontent.com/abagilet12/'
       'An-lisis-electoral---Regi-n-Centro-2003---2023/'
       'claude/analisis-electoral-centro-tp8vwh/'
       'datos/procesados/nomenclador_circuitos_2023.csv')

def codigos_conocidos(anio):
    """Codigos ya relevados, para no redescubrirlos."""
    if anio != '2023':
        return []
    try:
        with urllib.request.urlopen(RAW, timeout=30) as r:
            texto = r.read().decode('utf-8').splitlines()
        return [f['circuito_id'] for f in csv.DictReader(texto)]
    except Exception as e:
        print('no se pudieron leer los codigos conocidos:', e)
        return []


def circuitos_de(anio, te):
    """Codigos conocidos si los hay; si no, barrido."""
    conocidos = codigos_conocidos(anio)
    if conocidos:
        print(f'{anio}: {len(conocidos)} codigos conocidos, sin barrido')
        return conocidos
    return descubrir_circuitos(anio, te)


print(f'2023: {len(codigos_conocidos("2023"))} codigos sembrados')

## 5. Recolectar

Para cada elección se usa el nivel más fino que haya dado resultados:
circuito si existe, sección si no. La salida es una fila por unidad y
agrupación, con el mismo esquema que el resto de la base.

In [ ]:
def extraer_filas(anio, te, nombre, ambitos, nivel):
    """Convierte las respuestas de la API en filas tidy."""
    clave = 'circuito' if nivel == 'circuito' else 'seccion'
    respuestas = en_paralelo(
        lambda a: (a, consultar(anio, te, **{clave: a})),
        ambitos, f'{nombre} {anio}')

    filas = []
    for ambito, r in respuestas:
        if not tiene_datos(r):
            continue
        estado = r['estadoRecuento']
        base = {
            'anio': anio, 'instancia': nombre, 'cargo': 'PRESIDENTE Y VICE',
            'distrito_id': DISTRITO_ID, 'distrito': 'Santa Fe',
            'nivel': nivel,
            'seccion_id': ambito if nivel == 'seccion' else '',
            'circuito_id': ambito if nivel == 'circuito' else '',
            'mesas': estado['mesasTotalizadas'],
            'electores': estado['cantidadElectores'],
            'votantes': estado['cantidadVotantes'],
            'recuento_tipo': 'PROVISORIO', 'fuente': 'api_dine',
        }
        for a in r.get('valoresTotalizadosPositivos', []):
            filas.append({**base, 'agrupacion': a['nombreAgrupacion'],
                          'tipo_registro': 'agrupacion', 'votos': a['votos']})
            # En PASO cada agrupacion trae sus listas internas.
            for l in a.get('listas', []) or []:
                filas.append({**base, 'agrupacion': a['nombreAgrupacion'],
                              'lista': l.get('nombre', ''),
                              'tipo_registro': 'lista', 'votos': l['votos']})
        otros = r.get('valoresTotalizadosOtros') or {}
        for campo, tipo in [('votosNulos', 'nulos'),
                            ('votosEnBlanco', 'blancos'),
                            ('votosRecurridosComandoImpugnados', 'recurridos')]:
            if campo in otros:
                filas.append({**base, 'agrupacion': tipo,
                              'tipo_registro': tipo, 'votos': otros[campo]})
    return filas


todas = []
for anio, te, nombre in elecciones:
    circuitos = circuitos_de(anio, te)
    if circuitos:
        filas = extraer_filas(anio, te, nombre, circuitos, 'circuito')
    else:
        secciones = descubrir_secciones(anio, te)
        filas = extraer_filas(anio, te, nombre, secciones, 'seccion')
    todas += filas
    print(f'{anio} {nombre}: {len(filas)} filas\n')

## 6. Controles de consistencia

Sin esto no se publica nada. Se comprueba que las agrupaciones sumen los
positivos y que el padrón agregado sea coherente.

In [ ]:
import pandas as pd

df = pd.DataFrame(todas)

print('Filas por eleccion y nivel:')
print(df.groupby(['anio', 'instancia', 'nivel']).size().to_string(), '\n')

# El padron de una unidad se repite en cada una de sus filas: hay que
# quedarse con un valor por unidad antes de sumar, o se multiplica.
clave_unidad = ['anio', 'instancia', 'seccion_id', 'circuito_id']
unidades = df.drop_duplicates(clave_unidad)

resumen = []
for (anio, inst), g in df.groupby(['anio', 'instancia']):
    pos = g[g.tipo_registro == 'agrupacion'].votos.sum()
    u = unidades[(unidades.anio == anio) & (unidades.instancia == inst)]
    resumen.append({'anio': anio, 'instancia': inst,
                    'unidades': len(u), 'mesas': u.mesas.sum(),
                    'electores': u.electores.sum(),
                    'votantes': u.votantes.sum(), 'positivos': pos})
print(pd.DataFrame(resumen).to_string(index=False))

**Cómo leer el control.** Comparar `mesas` y `electores` contra lo que se
sabe de la provincia (Santa Fe 2023: 8.332 mesas, 2.827.794 electores). Si el
barrido dejó ámbitos afuera, acá se nota: los totales quedan cortos. En ese
caso hay que ampliar `RANGO_CIRCUITOS` y volver a correr.

## 7. Exportar

Un CSV por elección, más uno consolidado, guardados en Drive.

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    SALIDA = Path('/content/drive/MyDrive/analisis_electoral/api_dine')
except ImportError:
    SALIDA = Path('salida_api')      # fuera de Colab

SALIDA.mkdir(parents=True, exist_ok=True)

for (anio, inst), g in df.groupby(['anio', 'instancia']):
    destino = SALIDA / f'{anio}_{inst}_santa_fe_presidente_api.csv'
    g.to_csv(destino, index=False, encoding='utf-8')
    print(f'-> {destino.name} ({len(g)} filas)')

df.to_csv(SALIDA / 'api_dine_santa_fe_consolidado.csv', index=False,
          encoding='utf-8')
print(f'\n-> api_dine_santa_fe_consolidado.csv ({len(df)} filas)')

## 8. Qué hacer con esto

Subir los CSV al repositorio en `datos/crudos/api_dine/` y correr los parsers
del proyecto para integrarlos a la serie.

**Recordar al analizar:**

- Estos datos son **provisorios**. El nivel provincial de la base usa
  escrutinio definitivo. No mezclarlos en un mismo cálculo; para eso están
  las columnas `recuento_tipo` y `fuente`.
- Al sumar votos, filtrar `tipo_registro == 'agrupacion'`. Las listas
  internas de las PASO ya están contenidas en su agrupación.
- Los códigos de circuito **no son comparables entre años**: Santa Fe los
  renumeró en 2023.
- Una serie temporal exige además **el mismo universo geográfico** todos los
  años. Ver `docs/HOMOLOGACION.md`.